# Colab 6D Two Records


In [ ]:
# pd.merge(left, right, on='key')                       # the join pattern
# pd.merge(left, right, on='key', how='left')           # keep every row on the left

# pd.to_datetime(column, format='%Y-%m-%d')             # the parsing pattern
# column.dt.year                                        # and its accessors

# df.pivot_table(index=, columns=, values=)             # the pivot pattern

SyntaxError: invalid syntax (708280711.py, line 7)

In [ ]:
# df.groupby('key')['column'].agg(['count', 'mean'])
# df.sort_values('column', ascending=False).head(n)

In [ ]:
import pandas as pd

base = 'https://eds-217-essential-python.github.io/data/'

temp = pd.read_csv(base + 'monthly_temperature_data.csv')
co2 = pd.read_csv(base + 'monthly_co2_concentration.csv')

In [ ]:
temp.head()

,Date,MonthlyAnomaly
0,1880-01-01,-0.20
1,1880-02-01,-0.25
2,1880-03-01,-0.09
3,1880-04-01,-0.16
4,1880-05-01,-0.09


In [ ]:
co2.head()

,Date,CO2Concentration
0,1958-04-01,317.45
1,1958-05-01,317.51
2,1958-06-01,317.27
3,1958-07-01,315.87
4,1958-08-01,314.93


# Part 1: Making one table out of two

In [ ]:
co2.shape

(796, 2)

In [ ]:
temp.shape

(1736, 2)

How many rows does each table have, and what columns? What is the earliest and latest `Date` in each? (`.min()` and `.max()` on the `Date` column will do it, because these dates are text in `%Y-%m-%d` form and text in that form sorts correctly.)

In [ ]:
print(co2['Date'].min())
print(temp['Date'].min())

print(co2['Date'].max())
print(temp['Date'].max())

1958-04-01
1880-01-01
2024-07-01
2024-08-01


Merge the two tables on `Date`, with the default `how=`. How many rows come back?

In [ ]:
merged = pd.merge(co2, temp, on = 'Date', how = 'inner')
merged.shape

(796, 3)

Merge them again with `how='left'`, putting `temp` on the left. How many rows now, and how many nulls, in which column?

the 940 rows didnt have values for co2

In [ ]:
merged_left = pd.merge(temp, co2, on = 'Date', how = 'left')
merged_left.shape

(1736, 3)

In a markdown cell: the two merges differ by 940 rows. Provide a sentence explaining what those 940 rows contain (your answer to question 1 should already have this).

Suppose you are writing one of the following two papers. For each one, say which merge you would use and why:

“Atmospheric CO₂ and global temperature move together”
“The global surface temperature record, 1880 to the present”

# Part 2: Making the dates real

Copy the inner merge into a table called `climate`, ending the line with `.copy()`. Then parse its `Date` column into a new column called date. Demonstrate that the parse worked by printing the new column’s dtype.

In [ ]:
climate = pd.merge(co2, temp, on = 'Date', how= 'inner')
climate.head(1)

,Date,CO2Concentration,MonthlyAnomaly
0,1958-04-01,317.45,0.01


In [ ]:
climate['date'] = pd.to_datetime(climate['Date'], format = '%Y-%m-%d')

climate['date']

0     1958-04-01
1     1958-05-01
2     1958-06-01
3     1958-07-01
4     1958-08-01
         ...    
791   2024-03-01
792   2024-04-01
793   2024-05-01
794   2024-06-01
795   2024-07-01
Name: date, Length: 796, dtype: datetime64[ns]

In [ ]:
climate['date'].dtype

dtype('<M8[ns]')

Add `year` and `month` columns using the `.dt` accessors.

In [ ]:
climate['year']= climate['date'].dt.year
climate['month']= climate['date'].dt.month
climate['day']= climate['date'].dt.day

climate.head(2)

,Date,CO2Concentration,MonthlyAnomaly,date,year,month,day
0,1958-04-01,317.45,0.01,1958-04-01,1958,4,1
1,1958-05-01,317.51,0.06,1958-05-01,1958,5,1


Build a table of annual means: group by `year` and report the count of months, the mean of `MonthlyAnomaly`, and the mean of `CO2Concentration`. Show the first three rows and the last three rows.

In [ ]:
climate_means = pd.DataFrame({
    'month_counts': climate.groupby('year')['month'].count(),
    'average_ma': climate.groupby('year')['MonthlyAnomaly'].mean(),
    'averages_co2': climate.groupby('year')['CO2Concentration'].mean()})

print(climate_means.head(3))
print(climate_means.tail(3))

      month_counts  average_ma  averages_co2
year                                        
1958             9    0.004444    315.184444
1959            12    0.030833    315.981667
1960            12   -0.025000    316.908333
      month_counts  average_ma  averages_co2
year                                        
2022            12    0.893333    418.528333
2023            12    1.169167    421.075833
2024             7    1.287143    425.522857


Two of the sixty-seven years in the table are not twelve months long. Which two, and why? Filter the data into a new table called `full_years`. How many years survive?

In [ ]:
full_years = climate_means[climate_means['month_counts']== 12].reset_index()

full_years.head(1)

,year,month_counts,average_ma,averages_co2
0,1959,12,0.030833,315.981667


# Part 3: The size of the annual cycle

Build a wide table with `year` down the rows, `month` across the columns, and `CO2Concentration` in the cells. Show the first three rows and the last three rows.

In [51]:
wide_climate = climate.pivot_table(
    index = 'year',
    columns = 'month',
    values = 'CO2Concentration'
    )

wide_climate = wide_climate.reset_index()

The wide table holds two different patterns at once. Read it down a single column, then read it across a single row. In a markdown cell, describe both in one sentence each.

For every year, the difference between its largest monthly value and its smallest is the size of the annual cycle. Compute that difference for two specific years, 1959 and 2023, and report both. (You will need two lines of code; one for each year)

In [68]:
comparison = pd.DataFrame({
    'max': climate.groupby('year')['CO2Concentration'].max(),
    #'y1959max': y1959.groupby('year')['CO2Concentration'].max(),
    'min': climate.groupby('year')['CO2Concentration'].min()
    #'y2023max': y2023.groupby('year')['CO2Concentration'].max()
})

comparison['diff'] = comparison['max'] - comparison['min']
comparison = comparison.reset_index()


In [76]:
y1959 = comparison[comparison['year'] == 1959]
y2023 = comparison[comparison['year'] == 2023]
print(y1959['diff'])
print(y2023['diff'])

1    4.96
Name: diff, dtype: float64
65    5.5
Name: diff, dtype: float64


In a markdown cell: the annual cycle in CO₂ is caused by vegetation (mainly in the northern hemisphere), which removes carbon from the atmosphere during the growing season and releases it back in the autumn/winter. Given that context, explain what your two numbers from question 12 might indicate. Are two years sufficient to make that inference?

Build the same wide table for `MonthlyAnomaly`. Does the temperature record have a comparable seasonal cycle in it? Say why or why not in one sentence.

In [77]:
comparison_ma = pd.DataFrame({
    'max': climate.groupby('year')['MonthlyAnomaly'].max(),
    'min': climate.groupby('year')['MonthlyAnomaly'].min()
})

comparison_ma['diff'] = comparison_ma['max'] - comparison_ma['min']
comparison_ma = comparison_ma.reset_index()

In [78]:
y1959 = comparison_ma[comparison_ma['year'] == 1959]
y2023 = comparison_ma[comparison_ma['year'] == 2023]
print(y1959['diff'])
print(y2023['diff'])

1    0.26
Name: diff, dtype: float64
65    0.61
Name: diff, dtype: float64


# Part 4: Making a statement about change

Using `full_years`, compare the 1960s with the 2010s. Compute the mean of each column for `year` between 1960 and 1969, and again for `year` between 2010 and 2019, and report the change in each.

In [79]:
full_years.head(2)

,year,month_counts,average_ma,averages_co2
0,1959,12,0.030833,315.981667
1,1960,12,-0.025000,316.908333


In [91]:
the60s = full_years[(full_years['year'] >= 1960) & (full_years['year'] <= 1969)]
the2010s = full_years[(full_years['year'] >= 2010) & (full_years['year'] <= 2019)]
mean_60 = the60s['averages_co2'].mean()
mean_2010 = the2010s['averages_co2'].mean()
difference = mean_2010 - mean_60
difference

80.12341666666669

In a markdown cell of four or five sentences: state what these calculations indicate in terms of changes in the temperature and CO2 concentrations, with units. You have put two variables in the same table and found that both varied together; explain what a data scientist would need before making a causal claim about this covariation.